In [2]:
import copy
import time
from collections import deque

class SudokuCSP:
    def __init__(self, board):
        self.size = 9
        self.board = board
        self.domains = {}
        self.backtrack_calls = 0
        self.backtrack_failures = 0
        self.init_domains()
        self.ac3()
    
    def init_domains(self):
        for i in range(self.size):
            for j in range(self.size):
                if self.board[i][j] != 0:
                    self.domains[(i, j)] = [self.board[i][j]]
                else:
                    self.domains[(i, j)] = list(range(1, 10))
        
        for i in range(self.size):
            for j in range(self.size):
                if self.board[i][j] != 0:
                    self.domains[(i, j)] = [self.board[i][j]]
                    self.remove_from_neighbors(i, j, self.board[i][j])
    
    def remove_from_neighbors(self, row, col, val):
        for j in range(self.size):
            if j != col and val in self.domains[(row, j)]:
                self.domains[(row, j)].remove(val)
        for i in range(self.size):
            if i != row and val in self.domains[(i, col)]:
                self.domains[(i, col)].remove(val)
        box_row = (row // 3) * 3
        box_col = (col // 3) * 3
        for i in range(box_row, box_row + 3):
            for j in range(box_col, box_col + 3):
                if (i, j) != (row, col) and val in self.domains[(i, j)]:
                    self.domains[(i, j)].remove(val)
    
    def get_peers(self, row, col):
        peers = []
        for j in range(self.size):
            if j != col:
                peers.append((row, j))
        for i in range(self.size):
            if i != row:
                peers.append((i, col))
        box_row = (row // 3) * 3
        box_col = (col // 3) * 3
        for i in range(box_row, box_row + 3):
            for j in range(box_col, box_col + 3):
                if (i, j) != (row, col):
                    peers.append((i, j))
        temp = []
        for p in peers:
            if p not in temp:
                temp.append(p)
        return temp
    
    def ac3(self):
        queue = deque()
        for i in range(self.size):
            for j in range(self.size):
                for peer in self.get_peers(i, j):
                    queue.append(((i, j), peer))
        
        while queue:
            (xi, xj), (yi, yj) = queue.popleft()
            if self.revise((xi, xj), (yi, yj)):
                if len(self.domains[(xi, xj)]) == 0:
                    return False
                for peer in self.get_peers(xi, xj):
                    if peer != (yi, yj):
                        queue.append((peer, (xi, xj)))
        return True
    
    def revise(self, xi, xj):
        revised = False
        to_remove = []
        for x in self.domains[xi]:
            found = False
            for y in self.domains[xj]:
                if x != y:
                    found = True
                    break
            if not found:
                to_remove.append(x)
                revised = True
        for x in to_remove:
            self.domains[xi].remove(x)
        return revised
    
    def pick_variable(self, domains):
        best_var = None
        best_len = 10
        for var, domain in domains.items():
            if len(domain) > 1 and len(domain) < best_len:
                best_len = len(domain)
                best_var = var
        return best_var
    
    def check_value(self, var, value, domains):
        row, col = var
        for peer in self.get_peers(row, col):
            if len(domains[peer]) == 1 and domains[peer][0] == value:
                return False
        return True
    
    def all_assigned(self, domains):
        for domain in domains.values():
            if len(domain) != 1:
                return False
        return True
    
    def make_board(self, domains):
        board = [[0 for _ in range(self.size)] for _ in range(self.size)]
        for (i, j), domain in domains.items():
            board[i][j] = domain[0]
        return board
    
    def forward_check(self, var, value, domains):
        row, col = var
        pruned = []
        for peer in self.get_peers(row, col):
            if value in domains[peer]:
                domains[peer].remove(value)
                pruned.append((peer, value))
                if len(domains[peer]) == 0:
                    for p, v in pruned:
                        domains[p].append(v)
                    return None
        return pruned
    
    def undo_prune(self, pruned, domains):
        for var, val in pruned:
            domains[var].append(val)
            domains[var].sort()
    
    def search(self, domains):
        self.backtrack_calls = self.backtrack_calls + 1
        
        if self.all_assigned(domains):
            return domains
        
        var = self.pick_variable(domains)
        if var is None:
            return None
        
        row, col = var
        
        for value in sorted(domains[var]):
            if self.check_value(var, value, domains):
                new_domains = copy.deepcopy(domains)
                new_domains[var] = [value]
                
                pruned = self.forward_check(var, value, new_domains)
                if pruned is not None:
                    result = self.search(new_domains)
                    if result is not None:
                        return result
                    self.undo_prune(pruned, new_domains)
        
        self.backtrack_failures = self.backtrack_failures + 1
        return None
    
    def solve(self):
        start = time.time()
        result = self.search(self.domains)
        end = time.time()
        if result:
            return self.make_board(result), end - start
        return None, end - start
    
    def show(self, board):
        for i in range(9):
            if i % 3 == 0 and i != 0:
                print("-" * 21)
            for j in range(9):
                if j % 3 == 0 and j != 0:
                    print("|", end=" ")
                print(board[i][j] if board[i][j] != 0 else ".", end=" ")
            print()

def read_board(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                row = []
                for ch in line:
                    row.append(int(ch))
                board.append(row)
    return board

def run_test(name, filename):
    print()
    print("----------------------------------------")
    print(name)
    print("----------------------------------------")
    
    board = read_board(filename)
    print("Initial board:")
    temp = SudokuCSP(board)
    temp.show(board)
    
    solver = SudokuCSP(copy.deepcopy(board))
    solved, elapsed = solver.solve()
    
    if solved:
        print()
        print("Solution found!")
        solver.show(solved)
        print()
        print("Time:", round(elapsed, 4), "seconds")
        print("Backtrack calls:", solver.backtrack_calls)
        print("Backtrack failures:", solver.backtrack_failures)
        return True
    else:
        print()
        print("No solution found!")
        print("Time:", round(elapsed, 4), "seconds")
        print("Backtrack calls:", solver.backtrack_calls)
        print("Backtrack failures:", solver.backtrack_failures)
        return False

def main():
    print()
    print("SUDOKU CSP SOLVER")
    print()
    
    tests = [
        ("EASY", "easy.txt"),
        ("MEDIUM", "medium.txt"),
        ("HARD", "hard.txt"),
        ("VERY HARD", "veryhard.txt")
    ]
    
    all_results = []
    for name, filename in tests:
        success = run_test(name, filename)
        all_results.append((name, success))
    
    print()
    print("----------------------------------------")
    print("SUMMARY")
    print("----------------------------------------")
    for name, success in all_results:
        if success:
            print(name, ": SOLVED")
        else:
            print(name, ": FAILED")

if __name__ == "__main__":
    main()


SUDOKU CSP SOLVER


----------------------------------------
EASY
----------------------------------------
Initial board:
. . 4 | . 3 . | . 5 . 
6 . 9 | 4 . . | . . . 
. . 5 | 1 . . | 4 8 9 
---------------------
. . . | . 6 . | 9 3 . 
3 . . | 8 . 7 | . . 2 
. 2 6 | . 4 . | . . . 
---------------------
4 5 3 | . . 9 | 6 . . 
. . . | . . 4 | 7 . 5 
. 9 . | . 5 . | 2 . . 

Solution found!
7 8 4 | 9 3 2 | 1 5 6 
6 1 9 | 4 8 5 | 3 2 7 
2 3 5 | 1 7 6 | 4 8 9 
---------------------
5 7 8 | 2 6 1 | 9 3 4 
3 4 1 | 8 9 7 | 5 6 2 
9 2 6 | 5 4 3 | 8 7 1 
---------------------
4 5 3 | 7 2 9 | 6 1 8 
8 6 2 | 3 1 4 | 7 9 5 
1 9 7 | 6 5 8 | 2 4 3 

Time: 0.0 seconds
Backtrack calls: 1
Backtrack failures: 0

----------------------------------------
MEDIUM
----------------------------------------
Initial board:
. . . | 3 . . | 4 . . 
1 . 9 | 7 . . | . . . 
. . . | 8 5 1 | . 7 . 
---------------------
. . 2 | 6 . 7 | 8 3 . 
9 . 6 | . 1 . | 2 . 7 
. 3 1 | 5 . 2 | 9 . . 
---------------------
. 1 . | 3 6